In [ ]:
import pandas as pd
import os

# 경로 설정
path_score = r"C:\Users\cozy1\Documents\276_Scoring_Model\raw_data\PROD_FLOWSCORE\PUBLIC"

def audit_baseline_data():
    print("🔬 [Baseline Audit] 재무/비재무 기초 자산 정밀 점검")
    print("-" * 75)
    
    # 1. 재무 비율 점검
    try:
        ratio = pd.read_csv(os.path.join(path_score, "COMPANY_FINANCIAL_RATIO.csv"), encoding='utf-8-sig')
        print(f"📊 [재무] {len(ratio)}건의 재무 비율 데이터 확보")
        # 주요 비율 결측치 및 평균 확인
        cols = ['LIQUIDITY_RATIO', 'DEBT_RATIO', 'OPERATING_PROFIT_RATIO']
        existing_cols = [c for c in cols if c in ratio.columns]
        if existing_cols:
            display(ratio[existing_cols].describe().loc[['count', 'mean', '50%']])
    except: print("❌ 재무 비율 파일 로드 실패")

    # 2. 대표자 변경 히스토리 (비재무 안정성)
    try:
        rep_hist = pd.read_csv(os.path.join(path_score, "COMPANY_REPRESENTATIVE_HISTORY.csv"), encoding='utf-8-sig')
        # 기업당 평균 대표자 변경 횟수 계산
        change_counts = rep_hist.groupby('COMPANY_ID').size()
        print(f"\n👨‍💼 [비재무] 기업당 평균 대표자 변경 횟수: {change_counts.mean():.2f}회")
        print(f"⚠️ 3회 이상 변경된 불안정 기업: {len(change_counts[change_counts >= 3])}개")
    except: print("❌ 대표자 히스토리 파일 로드 실패")

    # 3. 고용 통계 (비재무 규모)
    try:
        emp = pd.read_csv(os.path.join(path_score, "COMPANY_EMPLOYEE_STATISTICS.csv"), encoding='utf-8-sig')
        print(f"\n👥 [비재무] 고용 데이터 {len(emp)}건 확보")
    except: print("❌ 고용 통계 파일 로드 실패")

# 실행
audit_baseline_data()

🔬 [Baseline Audit] 재무/비재무 기초 자산 정밀 점검
---------------------------------------------------------------------------
📊 [재무] 3580건의 재무 비율 데이터 확보

👨‍💼 [비재무] 기업당 평균 대표자 변경 횟수: 1.15회
⚠️ 3회 이상 변경된 불안정 기업: 49개

👥 [비재무] 고용 데이터 7881건 확보


In [ ]:
import pandas as pd
import os
import numpy as np

# 1. 경로 설정
path_score = r"C:\Users\cozy1\Documents\276_Scoring_Model\raw_data\output\PROD_FLOWSCORE\PUBLIC"
path_point = r"C:\Users\cozy1\Documents\276_Scoring_Model\raw_data\output\PROD_FLOWPOINT\PUBLIC"

# 2. 검사 대상 파일 및 핵심 식별자 정의
target_files = {
    'Financial_Ratio': (path_score, 'COMPANY_FINANCIAL_RATIO.csv', 'COMPANY_ID'),
    'Rep_History': (path_score, 'COMPANY_REPRESENTATIVE_HISTORY.csv', 'COMPANY_ID'),
    'Links': (path_score, 'COMPANY_LINKS.csv', 'COMPANY_ID'),
    'BNPL_Requests': (path_point, 'PAY_BNPL_REQUESTS.csv', 'COMPANY_ID'),
    'BNPL_Items': (path_point, 'PAY_BNPL_BUY_ITEMS.csv', 'PAY_BNPL_REQUEST_ID'),
    'BNPL_Comments': (path_point, 'PAY_BNPL_COMMENTS.csv', 'PAY_BNPL_REQUEST_ID'),
    'Upload_Files': (path_point, 'PAY_BNPL_UPLOAD_FILES.csv', 'PAY_BNPL_REQUEST_ID')
}

def run_identifier_audit():
    print("🔍 [Identifier Audit] 식별자 무결성 전수 조사 가동...")
    print("-" * 85)
    
    audit_results = []

    for label, (folder, file, id_col) in target_files.items():
        file_path = os.path.join(folder, file)
        
        if not os.path.exists(file_path):
            print(f"⚠️ 파일 없음: {file}")
            continue
            
        # 데이터 로드 (속도를 위해 샘플링 없이 타입 분석)
        df = pd.read_csv(file_path, encoding='utf-8-sig', low_memory=False)
        
        if id_col not in df.columns:
            print(f"❌ 컬럼 누락: {file} 내 {id_col} 없음")
            continue

        ids = df[id_col]
        
        # [검사항목 1] 데이터 타입 및 샘플 형태
        dtype = str(ids.dtype)
        sample_val = str(ids.iloc[0]) if not ids.empty else "N/A"
        
        # [검사항목 2] 결측치 및 중복도
        null_count = ids.isnull().sum()
        unique_count = ids.nunique()
        total_rows = len(df)
        
        # [검사항목 3] 포맷 이슈 (소수점 포함 여부 등)
        has_dot_zero = ids.astype(str).str.contains(r'\.0$').any()
        
        audit_results.append({
            'Category': label,
            'File': file,
            'ID_Column': id_col,
            'Dtype': dtype,
            'Sample': sample_val,
            'Nulls': null_count,
            'Unique_IDs': unique_count,
            'Total_Rows': total_rows,
            'Has_Dot_Zero': '⚠️ Yes' if has_dot_zero else '✅ No'
        })

    report_df = pd.DataFrame(audit_results)
    display(report_df)
    return report_df

# 실행
id_audit_report = run_identifier_audit()

🔍 [Identifier Audit] 식별자 무결성 전수 조사 가동...
-------------------------------------------------------------------------------------
⚠️ 파일 없음: COMPANY_LINKS.csv


,Category,File,ID_Column,Dtype,Sample,Nulls,Unique_IDs,Total_Rows,Has_Dot_Zero
0,Financial_Ratio,COMPANY_FINANCIAL_RATIO.csv,COMPANY_ID,int64,15,0,1237,3580,✅ No
1,Rep_History,COMPANY_REPRESENTATIVE_HISTORY.csv,COMPANY_ID,int64,1,0,1513,1735,✅ No
2,BNPL_Requests,PAY_BNPL_REQUESTS.csv,COMPANY_ID,float64,367.0,8,133,495,⚠️ Yes
3,BNPL_Items,PAY_BNPL_BUY_ITEMS.csv,PAY_BNPL_REQUEST_ID,int64,252,0,367,439,✅ No
4,BNPL_Comments,PAY_BNPL_COMMENTS.csv,PAY_BNPL_REQUEST_ID,int64,5,0,135,220,✅ No
5,Upload_Files,PAY_BNPL_UPLOAD_FILES.csv,PAY_BNPL_REQUEST_ID,int64,1,0,297,3930,✅ No


In [4]:
import pandas as pd
import os

def run_robust_id_audit(root_path='.'):
    print("🚀 [Full-Spectrum Audit] 모든 하위 폴더 CSV 전수 조사 시작...")
    print("-" * 90)
    
    audit_results = []
    
    # 1. 파일 시스템 전수 탐색
    for root, dirs, files in os.walk(root_path):
        for file in files:
            if file.endswith('.csv'):
                file_path = os.path.join(root, file)
                try:
                    # 속도를 위해 우선 5행만 읽어 컬럼명 파악
                    temp_df = pd.read_csv(file_path, nrows=5, encoding='utf-8-sig')
                    
                    # 식별자 후보 추출 로직 (ID, COMPANY, REQUEST, CORP 등 포함된 컬럼)
                    id_cols = [c for c in temp_df.columns if any(k in c.upper() for k in ['ID', 'COMPANY', 'REQUEST', 'CORP'])]
                    
                    if not id_cols: continue
                    
                    # 2. 식별자별 상세 무결성 검사
                    for col in id_cols:
                        # 메모리 효율을 위해 해당 컬럼만 로드
                        full_col = pd.read_csv(file_path, usecols=[col], encoding='utf-8-sig')[col]
                        
                        audit_results.append({
                            'Group': os.path.basename(root), # 폴더명을 그룹으로 추정
                            'FileName': file,
                            'ID_Column': col,
                            'Dtype': str(full_col.dtype),
                            'Unique_IDs': full_col.nunique(),
                            'Null_Count': full_col.isnull().sum(),
                            'Has_Float_Trap': full_col.astype(str).str.contains(r'\.0$').any(),
                            'Total_Rows': len(full_col)
                        })
                except Exception as e:
                    print(f"⚠️ [Error] {file} 처리 불가: {e}")

    # 3. 결과 요약 리포트
    if not audit_results:
        print("❌ 발견된 CSV 파일이 없거나 식별자 컬럼을 찾지 못했습니다. 경로를 다시 확인해주세요.")
        return None
        
    report = pd.DataFrame(audit_results)
    # 정합성이 깨지기 쉬운 순서대로 정렬
    report = report.sort_values(by=['Has_Float_Trap', 'Null_Count'], ascending=False)
    
    print(f"\n✅ 감사 완료: 총 {len(report)}개의 식별자 포인트 발견")
    return report

# 실행 (현재 디렉토리부터 하위로 무조건 검색)
final_audit_report = run_robust_id_audit(root_path='.')
if final_audit_report is not None:
    display(final_audit_report)

🚀 [Full-Spectrum Audit] 모든 하위 폴더 CSV 전수 조사 시작...
------------------------------------------------------------------------------------------

✅ 감사 완료: 총 790개의 식별자 포인트 발견


,Group,FileName,ID_Column,Dtype,Unique_IDs,Null_Count,Has_Float_Trap,Total_Rows
380,PUBLIC,PAYABLE_TRANSACTIONS.csv,CANCEL_ID,float64,8030,16678,True,24708
532,PUBLIC,RECEIVABLE_TRANSACTIONS.csv,CANCEL_ID,float64,8030,16678,True,24708
362,PUBLIC,PAYABLE_DETAILS.csv,LAST_PAYABLE_DETAIL_ID,float64,8491,6109,True,14601
363,PUBLIC,PAYABLE_DETAILS.csv,FIRST_PAYABLE_DETAIL_ID,float64,4997,6109,True,14601
506,PUBLIC,RECEIVABLE_DETAILS.csv,LAST_RECEIVABLE_DETAIL_ID,float64,8491,6105,True,14597
...,...,...,...,...,...,...,...,...
784,PUBLIC,USER_COMPANY_SEARCH_HISTORY.csv,_AIRBYTE_RAW_ID,object,1464,0,False,1464
785,PUBLIC,USER_COMPANY_SEARCH_HISTORY.csv,_AIRBYTE_GENERATION_ID,int64,1,0,False,1464
786,PUBLIC,USER_COMPANY_SEARCH_HISTORY.csv,ID,int64,1464,0,False,1464
787,PUBLIC,USER_COMPANY_SEARCH_HISTORY.csv,USER_ID,int64,16,0,False,1464


In [6]:
import pandas as pd
import os

# 1. 4개 부문 17개 핵심 타겟 파일 리스트
target_files = [
    'COMPANY_OVERVIEW.csv', 'COMPANY_FINANCIAL_RATIO.csv', 'COMPANY_FINANCIAL_STATEMENT.csv',
    'COMPANY_REPRESENTATIVE_HISTORY.csv', 'COMPANY_EMPLOYEE_STATISTICS.csv',
    'RECEIVABLES.csv', 'RECEIVABLE_DETAILS.csv', 'COMPANY_LINKS.csv',
    'BNPL_BUYERS.csv', 'BNPL_SELLERS.csv', 'BUSINESS_RELATIONS.csv',
    'PAY_BNPL_REQUESTS.csv', 'PAY_BNPL_BUY_ITEMS.csv', 'PAY_BNPL_TERMS.csv',
    'PAY_BNPL_COMMENTS.csv', 'PAY_BNPL_UPLOAD_FILES.csv', 'PAY_BNPL_REQUEST_EVALUATION_LOGS.csv'
]

def run_target_preprocessing_audit(root_path='.'):
    print("🚀 [Target Audit] 17개 핵심 파일 탐색 및 전처리 진단 시작...")
    print("-" * 100)
    
    results = []
    found_files = set()
    
    # 2. 하위 폴더 전체를 뒤져서 타겟 파일만 핀셋 점검
    for root, dirs, files in os.walk(root_path):
        for file in files:
            if file in target_files:
                found_files.add(file)
                file_path = os.path.join(root, file)
                try:
                    df = pd.read_csv(file_path, low_memory=False, encoding='utf-8-sig')
                    
                    # 조인 키 우선순위 탐색
                    id_col = None
                    if 'COMPANY_ID' in df.columns:
                        id_col = 'COMPANY_ID'
                    elif 'PAY_BNPL_REQUEST_ID' in df.columns:
                        id_col = 'PAY_BNPL_REQUEST_ID'
                    elif 'ID' in df.columns:
                        id_col = 'ID'
                        
                    if id_col:
                        ids = df[id_col]
                        total_rows = len(df)
                        unique_ids = ids.nunique()
                        null_count = ids.isnull().sum()
                        # Float Trap (소수점 .0) 검사
                        has_float = ids.astype(str).str.contains(r'\.0$').any()
                        
                        # 1:1 마스터인지 1:N 트랜잭션인지 판별
                        relation = "1:1 (Master)" if unique_ids == total_rows else "1:N (Tx)"
                        
                        results.append({
                            'File': file,
                            'Key_Column': id_col,
                            'Dtype': str(ids.dtype),
                            'Nulls': null_count,
                            'Float_Trap': '⚠️ Yes' if has_float else '✅ No',
                            'Relation': relation,
                            'Unique_Keys': unique_ids,
                            'Total_Rows': total_rows
                        })
                except Exception as e:
                    print(f"⚠️ {file} 읽기 에러: {e}")

    # 3. 결과 출력
    if results:
        report = pd.DataFrame(results)
        # 위험한 파일(결측치 있거나 Float Trap 있는 파일)이 위로 오도록 정렬
        report = report.sort_values(by=['Float_Trap', 'Nulls'], ascending=[False, False])
        print("\n📊 [17개 파일 전처리 점검 결과표]")
        print(report.to_string(index=False))
    else:
        print("\n❌ 해당 경로에서 타겟 파일을 한 개도 찾지 못했습니다.")

    # 4. 누락된 파일 안내
    missing = set(target_files) - found_files
    if missing:
        print(f"\n⚠️ 발견되지 않은 파일 ({len(missing)}개) - 모델링에서 제외되거나 확인 필요:")
        for m in missing:
            print(f" - {m}")

# 주석 해제 완료. 즉시 실행됩니다.
run_target_preprocessing_audit('.')

🚀 [Target Audit] 17개 핵심 파일 탐색 및 전처리 진단 시작...
----------------------------------------------------------------------------------------------------

📊 [17개 파일 전처리 점검 결과표]
                                File          Key_Column   Dtype  Nulls Float_Trap     Relation  Unique_Keys  Total_Rows
                     BNPL_BUYERS.csv                  ID   int64      0       ✅ No 1:1 (Master)          133         133
                    BNPL_SELLERS.csv                  ID   int64      0       ✅ No 1:1 (Master)          138         138
              BUSINESS_RELATIONS.csv                  ID  object      0       ✅ No 1:1 (Master)            0           0
                   COMPANY_LINKS.csv          COMPANY_ID   int64      0       ✅ No     1:N (Tx)          874       25036
              PAY_BNPL_BUY_ITEMS.csv PAY_BNPL_REQUEST_ID   int64      0       ✅ No     1:N (Tx)          367         439
               PAY_BNPL_COMMENTS.csv          COMPANY_ID   int64      0       ✅ No     1:N (Tx)          